# Step 01 – Sentinel-2 composite: scene list and observation quality

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 2.2.1 · Table S1, Table S1b, Figure S1
- **Needs:** GEE exports `P2_R2_2_scene_list.csv` and `P2_R2_3_observations.tif` in `DATA_DIR`
- **Writes:** `RESULTS_DIR/composite_docs/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / P2_R2_2_scene_list.csv
#   DATA_DIR / P2_R2_3_observations.tif
#   RESULTS_DIR / composite_docs/

In [ ]:
!pip install -q rasterio

In [ ]:
"""
Summary table and observation maps from the GEE exports

Run after the two GEE exports (P2_R2_2_scene_list.csv, P2_R2_3_observations.tif) are in DATA_DIR.
In Colab first:  !pip install -q rasterio

Outputs (OUT_DIR):
  R2_2_scenes_per_tile.csv      main-text summary table (per MGRS tile)
  R2_2_scene_list_full.csv      full list for the Supplementary Materials (sorted)
  R2_3_observation_stats.csv    statistics of observations per pixel
  Fig_R2_3_observations.(png|pdf)
"""


import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

SRC_DIR = DATA_DIR
OUT_DIR = RESULTS_DIR + 'composite_docs/'
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ scene list per MGRS tile
sc = pd.read_csv(SRC_DIR + 'P2_R2_2_scene_list.csv', dtype={'processing_baseline': str})   # keep '05.11'
sc['date'] = pd.to_datetime(sc['date'])
sc = sc.sort_values(['mgrs_tile', 'date'])
sc.to_csv(OUT_DIR + 'R2_2_scene_list_full.csv', index=False)

per_tile = sc.groupby('mgrs_tile').agg(
    scenes=('product_id', 'count'),
    first_date=('date', 'min'), last_date=('date', 'max'),
    months_covered=('date', lambda d: d.dt.month.nunique()),
    mean_cloudy_pct=('cloudy_pixel_percentage', 'mean'),
    sentinel_2A=('spacecraft', lambda s: int((s == 'Sentinel-2A').sum())),
    sentinel_2B=('spacecraft', lambda s: int((s == 'Sentinel-2B').sum())),
    sentinel_2C=('spacecraft', lambda s: int((s == 'Sentinel-2C').sum())),
).reset_index()
per_tile['first_date'] = per_tile['first_date'].dt.strftime('%Y-%m-%d')
per_tile['last_date'] = per_tile['last_date'].dt.strftime('%Y-%m-%d')
per_tile['mean_cloudy_pct'] = per_tile['mean_cloudy_pct'].round(2)
per_tile = per_tile.loc[:, (per_tile != 0).any()]          # drop empty spacecraft columns
per_tile.to_csv(OUT_DIR + 'R2_2_scenes_per_tile.csv', index=False)
print(f'Scenes: {len(sc)} scenes, {sc["mgrs_tile"].nunique()} tiles, '
      f'{sc["date"].min():%Y-%m-%d} to {sc["date"].max():%Y-%m-%d}')
print('Processing baselines:', sorted(sc['processing_baseline'].astype(str).unique()))
print('\n', per_tile.to_string(index=False))

# Scenes per month (all tiles) - shows the seasonal distribution of the composite
by_month = sc.groupby(sc['date'].dt.month)['product_id'].count().reindex(range(1, 13), fill_value=0)
print('\nScenes per month:', by_month.to_dict())

# ------------------------------------------------------------------ observations per pixel
with rasterio.open(SRC_DIR + 'P2_R2_3_observations.tif') as src:
    n_obs, n_clear = src.read(1).astype(float), src.read(2).astype(float)
    b = src.bounds
inside = n_obs > 0
stats = []
for name, a in [('n_obs (scenes used)', n_obs), ('n_clear (SCL clear)', n_clear)]:
    v = a[inside]
    stats.append({'layer': name, 'min': v.min(), 'p5': np.percentile(v, 5), 'median': np.median(v),
                  'mean': v.mean(), 'max': v.max(),
                  '% pixels < 5 obs': 100 * (v < 5).mean(), '% pixels < 10 obs': 100 * (v < 10).mean()})
share = n_clear[inside] / n_obs[inside]
stats.append({'layer': 'clear share (n_clear / n_obs)', 'min': share.min(), 'p5': np.percentile(share, 5),
              'median': np.median(share), 'mean': share.mean(), 'max': share.max(),
              '% pixels < 5 obs': np.nan, '% pixels < 10 obs': np.nan})
stats = pd.DataFrame(stats).round(2)
stats.to_csv(OUT_DIR + 'R2_3_observation_stats.csv', index=False)
print('\nR2.3 observations per pixel (100 m grid, study area only):\n', stats.to_string(index=False))

ext = [b.left / 1000, b.right / 1000, b.bottom / 1000, b.top / 1000]
vmax = np.nanpercentile(n_obs[inside], 99)
fig, axes = plt.subplots(1, 2, figsize=(7.1, 4.3), sharey=True)
for ax, a, title in [(axes[0], n_obs, 'Scenes used in the median'),
                     (axes[1], n_clear, 'Clear observations (L2A SCL)')]:
    im = ax.imshow(np.ma.masked_where(~inside, a), cmap='YlOrBr', vmin=0, vmax=vmax, extent=ext)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Easting (km)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_aspect('equal')
axes[0].set_ylabel('Northing (km)', fontsize=8)
cb = fig.colorbar(im, ax=axes, shrink=0.8)
cb.set_label('Observations per pixel (2025)', fontsize=8)
cb.ax.tick_params(labelsize=7)
fig.savefig(OUT_DIR + 'Fig_R2_3_observations.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig_R2_3_observations.pdf', bbox_inches='tight')
plt.show()
print('\nFiles saved to:', OUT_DIR)